# 06 — BQ4: Come influiscono le caratteristiche del corso sulla retention?

> **Notebook 06 di 7** | Learning Retention Analytics  
> Analisi della quarta business question: confronto delle caratteristiche di progettazione dei corsi e relazione con la retention degli studenti.

## Scopo e ambito

Questo notebook risponde a **BQ4: Come influiscono le caratteristiche del corso sulla retention?** — la quarta delle cinque business question che guidano il progetto.

I notebook precedenti si sono concentrati sull'analisi a livello di studente: quando gli studenti abbandonano (BQ1), quali comportamenti precoci predicono il dropout (BQ2) e se i dati demografici o il comportamento predicono l'esito in modo più forte (BQ3). Questo notebook **sposta l'unità di analisi dagli studenti ai corsi**.

**Cosa copre questo notebook:**
- Classificazione dei 7 moduli OULAD per tasso di completamento
- Esplorazione delle associazioni tra caratteristiche di progettazione del corso (durata, densità delle valutazioni, diversità delle risorse) e retention
- Confronto dell'intensità di engagement tra i corsi
- Identificazione di pattern a livello di corso attraverso una heatmap di design standardizzata
- Formulazione di ipotesi su quali caratteristiche del corso possano supportare la retention

**Cosa questo notebook NON fa:**
- Nessuna inferenza confermativa. Con solo 7 moduli, qualsiasi sintesi di correlazione o soglia è euristica descrittiva, non test di significatività formale. Tutta l'analisi è esplorativa.
- Nessuna affermazione causale. Le differenze nella retention potrebbero essere guidate dalla selezione degli studenti, dalla difficoltà della materia o da fattori non misurati — non solo dalla progettazione del corso.

**Cosa viene dopo:**
- **Notebook 07** (`07_bq5_recommendations_synthesis.ipynb`): sintesi dei risultati BQ1–BQ4 nelle 3 principali raccomandazioni operative (BQ5).

> **Trasferibilità metodologica:** Il confronto delle metriche di retention a livello di corso è analogo all'analisi del churn a livello di prodotto o piano in SaaS: quale tier di prodotto trattiene meglio? Quali caratteristiche del piano (durata del trial, set di funzionalità, flusso di onboarding) correlano con un churn più basso? L'approccio — classificazione descrittiva, profilazione delle feature, confronto tramite heatmap — si trasferisce direttamente.

## Indice

1. [Configurazione ambiente](#1.-Environment-Setup)
2. [Il dataset di analisi](#2.-The-Analysis-Dataset)
3. [Classificazione dei corsi per completamento](#3.-Course-Ranking-by-Completion)
4. [Caratteristiche di progettazione del corso vs completamento](#4.-Course-Design-Features-vs-Completion)
5. [Intensità di engagement per corso](#5.-Engagement-Intensity-by-Course)
6. [Heatmap di progettazione del corso](#6.-Course-Design-Heatmap)
7. [Ipotesi e limitazioni](#7.-Hypotheses-and-Limitations)
8. [Conclusioni chiave e prossimi passi](#8.-Key-Takeaways-and-Next-Steps)

---

**Prerequisiti:**
- La pipeline ETL deve essere stata eseguita: `python -m run_pipeline`
- Il database DuckDB in `data/db/oulad.duckdb` deve contenere tutte e 5 le viste analitiche

**Dataset:** Open University Learning Analytics Dataset (OULAD) — ~32K studenti, 7 corsi, clickstream comportamentale completo. Licenza: CC-BY 4.0.

## 1. Configurazione ambiente

Configuriamo gli import, i parametri di visualizzazione e le funzioni helper riutilizzabili.

**Note tecniche per il lettore:**
- Tutte le query al database passano attraverso `src.db.connection.execute_query()` — il livello di astrazione DB del progetto (ADR-003).
- La query SQL principale di BQ4 risiede in `sql/queries/q_bq4_course_comparison.sql` e viene caricata a runtime dal disco. La query aggrega metriche a livello di corso da `v_course_profile`, `v_engagement_daily` e `v_engagement_early`.
- Nessuna funzione di test statistico è importata — con solo 7 data point, tutta l'analisi è descrittiva.
- Le figure vengono salvate in `reports/figures/` a 150 DPI.

In [ ]:
# --- Path setup ---
# Notebooks live in notebooks/ but project modules are at the project root.
# We search upward for pyproject.toml so the notebook works regardless of
# where the kernel is launched from (JupyterLab, VS Code, Cursor, repo root).
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Walk upward from start until we find pyproject.toml (the repo root marker)."""
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError(
        f"Could not locate project root: no pyproject.toml found in '{start}' "
        "or any parent directory. Run the notebook from within the repository."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- Standard library ---
import logging
import warnings

# --- Third-party ---
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# --- Project modules ---
from src.config import FIGURES_DIR, QUERIES_DIR
from src.db.connection import execute_query

# --- Configuration ---
warnings.filterwarnings('ignore', category=FutureWarning)
logging.basicConfig(level=logging.WARNING)

# --- Visualization defaults ---
sns.set_theme(style='whitegrid', font_scale=1.1)

# Course-level palette: one color per module for consistent identification.
# 7 OULAD modules use tab10 for maximum visual distinction.
_MODULE_ORDER = ['AAA', 'BBB', 'CCC', 'DDD', 'EEE', 'FFF', 'GGG']
_TAB10 = plt.cm.tab10.colors
PALETTE_COURSE = {m: _TAB10[i] for i, m in enumerate(_MODULE_ORDER)}

# Shared axis labels — defined as constants to avoid
# duplicated string literals flagged by static analysis
LABEL_COMPLETION_RATE = 'Completion rate (%)'
LABEL_WITHDRAWAL_RATE = 'Withdrawal rate (%)'
LABEL_NUM_ENROLLMENTS = 'Number of enrollments'

FIG_DPI = 150
FIG_SIZE = (10, 6)
FIG_SIZE_WIDE = (16, 5)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name: str) -> None:
    """Save figure to reports/figures/ with consistent settings."""
    path = FIGURES_DIR / f'{name}.png'
    fig.savefig(path, dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
    print(f'  Saved: {path}')


# --- Load BQ4 query from SQL file ---
_bq4_sql_path = QUERIES_DIR / 'q_bq4_course_comparison.sql'
BQ4_SQL = _bq4_sql_path.read_text(encoding='utf-8')
print(f'Loaded BQ4 query from: {_bq4_sql_path.name} ({len(BQ4_SQL):,} chars)')

# --- Prerequisite check ---
# BQ4 query uses three views: v_course_profile (FROM), v_engagement_daily
# and v_engagement_early (scalar subqueries for engagement intensity).
try:
    _check_cp = execute_query('SELECT COUNT(*) AS n FROM v_course_profile')
    _check_daily = execute_query('SELECT COUNT(*) AS n FROM v_engagement_daily')
    _check_early = execute_query('SELECT COUNT(*) AS n FROM v_engagement_early')
    _n_cp = _check_cp['n'].iloc[0]
    _n_daily = _check_daily['n'].iloc[0]
    _n_early = _check_early['n'].iloc[0]
    if _n_cp == 0 or _n_daily == 0 or _n_early == 0:
        raise RuntimeError('One or more views are empty')
    print('Database OK')
    print(f'  v_course_profile:    {_n_cp:>12,} rows')
    print(f'  v_engagement_daily:  {_n_daily:>12,} rows')
    print(f'  v_engagement_early:  {_n_early:>12,} rows')
except Exception as exc:
    raise RuntimeError(
        'Cannot query analytical views. '
        "Run 'python -m run_pipeline' first to populate the database."
    ) from exc

## 2. Il dataset di analisi

La query BQ4 (`q_bq4_course_comparison.sql`) aggrega metriche a livello di corso da `v_course_profile`, arricchite con l'intensità di engagement da `v_engagement_daily` e `v_engagement_early`. Ogni riga rappresenta **un modulo** (aggregato su tutte le sue presentazioni):

| Categoria | Colonne | Descrizione |
|-----------|---------|-------------|
| **Scala** | n_presentations, total_enrolled, total_completed | Quante coorti, studenti e completamenti |
| **Esiti** | avg_completion_rate_pct, avg_withdrawal_rate_pct | Performance di retention |
| **Progettazione del corso** | avg_course_length_days, avg_n_assessments, avg_assessment_density, avg_n_vle_resources, avg_n_activity_types | Caratteristiche controllate dal progettista del corso |
| **Engagement** | avg_clicks_per_student_day, median_early_clicks *(solo studenti attivi)* | Quanto intensamente gli studenti interagiscono con il corso |

**Avvertenza sulla metrica:** `median_early_clicks` proviene da `v_engagement_early` e riflette la mediana tra gli studenti con attività VLE registrata nei giorni 0–28. Gli studenti iscritti con zero click precoci sono esclusi, quindi questo valore può essere superiore alla mediana calcolata su tutti gli studenti iscritti.

**Decisione chiave di design:** Aggregare per `code_module` (non `code_module + code_presentation`) fornisce una vista stabile per corso. Le singole presentazioni possono variare, ma il design del modulo (valutazioni, risorse, struttura) è la variabile controllabile.

**Avvertenza critica:** Con solo **7 data point** (uno per modulo), questa analisi è interamente descrittiva. Nessun test statistico può rilevare effetti in modo affidabile con questa dimensione campionaria. I pattern sono ipotesi, non conclusioni.

In [ ]:
# --- Load BQ4 analysis dataset ---
df_bq4 = execute_query(BQ4_SQL)

modules_str = ', '.join(df_bq4['code_module'].tolist())
total_enrolled = df_bq4['total_enrolled'].sum()
total_presentations = df_bq4['n_presentations'].sum()

print(f'BQ4 dataset: {len(df_bq4)} rows (one per module) x {df_bq4.shape[1]} columns')
print(f'Modules: {modules_str}')
print(f'Total enrollments across all modules: {total_enrolled:,}')
print(f'Total presentations: {total_presentations}')

# --- Feature column constants ---
DESIGN_FEATURES = [
    'avg_course_length_days', 'avg_n_assessments',
    'avg_assessment_density', 'avg_n_vle_resources', 'avg_n_activity_types',
]
DESIGN_LABELS = {
    'avg_course_length_days': 'Course length (days)',
    'avg_n_assessments': 'Number of assessments',
    'avg_assessment_density': 'Assessment density (per 30d)',
    'avg_n_vle_resources': 'VLE resources',
    'avg_n_activity_types': 'Activity types',
}

ENGAGEMENT_FEATURES = ['avg_clicks_per_student_day', 'median_early_clicks']
ENGAGEMENT_LABELS = {
    'avg_clicks_per_student_day': 'Avg clicks per student-day',
    'median_early_clicks': 'Median early clicks (active students, first 28d)',
}

# --- Missingness check ---
n_nulls = df_bq4.isnull().sum()
if n_nulls.any():
    print()
    print('=== Null Values Detected ===')
    print(n_nulls[n_nulls > 0].to_string())
else:
    print()
    print('No null values — all 7 modules have complete data.')

# --- Full dataset overview ---
print()
print('=== Course Comparison Dataset (sorted by completion rate) ===')
print()
print(df_bq4.to_string(index=False))

> **Primo sguardo:** I 7 moduli OULAD coprono un'ampia gamma di tassi di completamento e design dei corsi. Alcuni moduli sono più brevi con meno valutazioni; altri sono più lunghi con più risorse e maggiore densità di valutazioni. La domanda è se queste differenze di design si relazionano con la performance di retention.
>
> Il dataset è volutamente piccolo — una riga per modulo — il che significa che ogni osservazione conta e nessun outlier può essere ignorato. Tutti i pattern che identifichiamo sono ipotesi che necessiterebbero di validazione su scala più ampia.

## 3. Classificazione dei corsi per completamento

Prima di esplorare le feature di design, stabiliamo la baseline: quali corsi trattengono meglio gli studenti?

Il tasso di completamento è mediato su tutte le presentazioni di ciascun modulo. Questo attenua la variazione specifica della coorte e si concentra sulle caratteristiche di retention intrinseche del modulo.

In [ ]:
# --- Course completion ranking ---
# Horizontal bar chart sorted by avg_completion_rate_pct.
# Module colors from PALETTE_COURSE provide visual consistency across figures.
df_ranked = df_bq4.sort_values(
    'avg_completion_rate_pct', ascending=True
).reset_index(drop=True)

fig, ax = plt.subplots(figsize=FIG_SIZE)

y_pos = np.arange(len(df_ranked))
colors = [PALETTE_COURSE[m] for m in df_ranked['code_module']]

ax.barh(y_pos, df_ranked['avg_completion_rate_pct'], color=colors, edgecolor='white')

# Annotate each bar with completion rate and enrollment volume
for i, (_, row) in enumerate(df_ranked.iterrows()):
    rate = row['avg_completion_rate_pct']
    enrolled = int(row['total_enrolled'])
    ax.text(
        rate + 0.8, i,
        f'{rate:.1f}%  (n={enrolled:,})',
        va='center', fontsize=9, color='#333333',
    )

# True overall completion rate from raw counts across all presentations
# (total_completed / total_enrolled), not a simple/unweighted average of module-level averages
overall_rate = (
    df_bq4['total_completed'].sum() / df_bq4['total_enrolled'].sum() * 100
)
ax.axvline(
    x=overall_rate, color='gray', linestyle='--', linewidth=1,
    label=f'Overall rate: {overall_rate:.1f}%',
)

ax.set_yticks(y_pos)
ax.set_yticklabels(df_ranked['code_module'])
ax.set_xlabel(LABEL_COMPLETION_RATE)
ax.set_title('Course Completion Rate Ranking\n(averaged across presentations)')
ax.legend(loc='lower right')
ax.set_xlim(0, 100)
sns.despine()
fig.tight_layout()
save_fig(fig, '06_course_completion_ranking')
plt.show()

# --- Quantify the spread ---
best = df_ranked.iloc[-1]
worst = df_ranked.iloc[0]
gap = best['avg_completion_rate_pct'] - worst['avg_completion_rate_pct']
best_rate = best['avg_completion_rate_pct']
worst_rate = worst['avg_completion_rate_pct']
best_module = best['code_module']
worst_module = worst['code_module']
print()
print(f'Completion rate range: {worst_rate:.1f}% ({worst_module}) '
      f'to {best_rate:.1f}% ({best_module}) — gap of {gap:.1f} pp')

> **Risultato chiave:** Il gap nel tasso di completamento tra i moduli con le migliori e le peggiori performance è sostanziale. Questa variazione non è casuale — persiste attraverso più presentazioni dello stesso modulo, suggerendo che qualcosa del corso stesso (design, difficoltà della materia, selezione degli studenti) guida la differenza.
>
> **Nota sul volume di iscrizioni:** L'annotazione mostra il totale delle iscrizioni per ogni modulo. Un numero maggiore di iscrizioni non correla necessariamente con un completamento più alto o più basso — il pattern è più sfumato, e lo esploreremo attraverso le feature di design nella prossima sezione.

## 4. Caratteristiche di progettazione del corso vs completamento

Le caratteristiche di progettazione del corso correlano con la retention? Esaminiamo due dimensioni chiave del design:

- **Durata del corso** (giorni): La durata influisce sul fatto che gli studenti restino iscritti?
- **Densità delle valutazioni** (valutazioni per 30 giorni): Valutazioni più frequenti aiutano o ostacolano la retention?

Ogni scatter plot mostra i 7 moduli posizionati per la loro feature di design (asse x) e tasso di completamento (asse y). La dimensione del punto è proporzionale al totale delle iscrizioni per contesto aggiuntivo.

**Promemoria:** Con n=7, cerchiamo *pattern visivi*, non relazioni statistiche. Qualsiasi tendenza apparente è un'ipotesi da testare con più dati.

In [ ]:
# --- Course design features vs completion rate ---
# 1x2 scatter panel: course length and assessment density vs completion.
# Bubble size proportional to enrollment volume for context.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_SIZE_WIDE)

# Scale enrollment to bubble sizes (100–600 pixel range)
enrolled = df_bq4['total_enrolled'].values
size_scale = 100 + 500 * (enrolled - enrolled.min()) / max(enrolled.max() - enrolled.min(), 1)

# --- Left: course length vs completion ---
for i, (_, row) in enumerate(df_bq4.iterrows()):
    ax1.scatter(
        row['avg_course_length_days'], row['avg_completion_rate_pct'],
        color=PALETTE_COURSE[row['code_module']],
        s=size_scale[i], edgecolor='white', linewidth=1.5, zorder=3,
    )
    ax1.annotate(
        row['code_module'],
        (row['avg_course_length_days'], row['avg_completion_rate_pct']),
        fontsize=9, fontweight='bold', ha='center', va='bottom',
        xytext=(0, 8), textcoords='offset points',
    )
ax1.set_xlabel(DESIGN_LABELS['avg_course_length_days'])
ax1.set_ylabel(LABEL_COMPLETION_RATE)
ax1.set_title('Course Length vs Completion Rate')
sns.despine(ax=ax1)

# --- Right: assessment density vs completion ---
for i, (_, row) in enumerate(df_bq4.iterrows()):
    ax2.scatter(
        row['avg_assessment_density'], row['avg_completion_rate_pct'],
        color=PALETTE_COURSE[row['code_module']],
        s=size_scale[i], edgecolor='white', linewidth=1.5, zorder=3,
    )
    ax2.annotate(
        row['code_module'],
        (row['avg_assessment_density'], row['avg_completion_rate_pct']),
        fontsize=9, fontweight='bold', ha='center', va='bottom',
        xytext=(0, 8), textcoords='offset points',
    )
ax2.set_xlabel(DESIGN_LABELS['avg_assessment_density'])
ax2.set_ylabel(LABEL_COMPLETION_RATE)
ax2.set_title('Assessment Density vs Completion Rate')
sns.despine(ax=ax2)

fig.suptitle(
    'Course Design Features vs Completion Rate\n'
    '(bubble size = total enrollment)',
    fontsize=14, y=1.02,
)
fig.tight_layout()
save_fig(fig, '06_course_design_vs_completion')
plt.show()

> **Pattern visivi:**
> - **Durata del corso:** Lo scatter potrebbe suggerire una relazione tra durata e retention — corsi più brevi o più lunghi potrebbero avere performance diverse. Tuttavia, con 7 punti, qualsiasi tendenza apparente potrebbe essere guidata da uno o due outlier.
> - **Densità delle valutazioni:** Corsi con pacing diverso delle valutazioni potrebbero mostrare profili di retention diversi. Se valutazioni più frequenti aiutano (attraverso checkpoint regolari di engagement) o ostacolano (attraverso affaticamento da valutazione) è una domanda aperta.
>
> **Fattori confondenti:** Queste associazioni sono confuse dalla difficoltà della materia, dall'autoselezione degli studenti e dalla qualità dei contenuti del modulo. Un corso con alta densità di valutazioni che capita anche di trattare una materia popolare e ben insegnata tratterrà gli studenti indipendentemente dalla frequenza delle valutazioni.

In [ ]:
# --- Exploratory rank correlations ---
# With only 7 data points, formal significance testing is meaningless
# (critical Spearman ρ for n=7 at α=0.05 two-tailed ≈ 0.79).
# These correlations serve as exploratory pattern summaries, not evidence.
all_features = DESIGN_FEATURES + ENGAGEMENT_FEATURES
all_labels = {**DESIGN_LABELS, **ENGAGEMENT_LABELS}

rho_series = (
    df_bq4[all_features + ['avg_completion_rate_pct']]
    .corr(method='spearman')['avg_completion_rate_pct']
    .drop('avg_completion_rate_pct')
)

print('=== Exploratory Spearman ρ vs Completion Rate (n=7) ===')
print('  CAUTION: 7 data points → descriptive patterns only, not statistical evidence.')
print('  |ρ| > 0.79 needed for significance at α=0.05 (two-tailed).')
print()

for feat in all_features:
    rho = rho_series[feat]
    label = all_labels[feat]
    if abs(rho) >= 0.79:
        direction = '★ notable'
    elif abs(rho) >= 0.4:
        direction = '~ moderate'
    else:
        direction = '— weak'
    print(f'  {label:35s} ρ = {rho:+.3f}  ({direction})')

> **Riepilogo delle correlazioni:** Le correlazioni di rango di Spearman forniscono un riepilogo compatto di quali feature *si muovono nella stessa direzione* del tasso di completamento. Le feature contrassegnate come notevoli (|ρ| ≥ 0.79) supererebbero la soglia di significatività per n=7, ma anche queste dovrebbero essere trattate come ipotesi.
>
> **Cosa le correlazioni non possono dirci:** Le correlazioni di rango misurano associazione monotona, non causalità. Un forte ρ positivo tra densità delle valutazioni e tasso di completamento potrebbe significare che le valutazioni aiutano la retention — o che i corsi popolari e ben progettati capita che abbiano più valutazioni. I dati non possono distinguere queste spiegazioni con 7 osservazioni.

## 5. Intensità di engagement per corso

Gli studenti interagiscono in modo diverso con corsi diversi? Due metriche complementari catturano l'intensità di engagement:

- **Click medi per studente-giorno** (da `v_engagement_daily`): l'intensità tipica di interazione giornaliera con il VLE. Valori più alti significano che gli studenti che accedono tendono a cliccare di più.
- **Click precoci mediani** (da `v_engagement_early`, solo studenti attivi): i click totali mediani nei primi 28 giorni tra gli studenti che hanno avuto almeno un'interazione VLE. Questo cattura il burst iniziale di engagement ed è meno sensibile agli outlier rispetto alla media. Gli studenti con zero click precoci sono esclusi (vedi avvertenza sulla metrica nella Sezione 2).

I moduli sono ordinati per tasso di completamento (stesso ordine del grafico di classificazione) per facilitare il confronto visivo.

In [ ]:
# --- Engagement intensity by course ---
# 1x2 panel: daily engagement intensity + early engagement volume.
# Modules sorted by completion rate for consistent reading.
df_eng = df_bq4.sort_values(
    'avg_completion_rate_pct', ascending=True
).reset_index(drop=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_SIZE_WIDE)

y_pos = np.arange(len(df_eng))
colors = [PALETTE_COURSE[m] for m in df_eng['code_module']]

# --- Left: avg clicks per student-day ---
ax1.barh(y_pos, df_eng['avg_clicks_per_student_day'], color=colors, edgecolor='white')
for i, (_, row) in enumerate(df_eng.iterrows()):
    val = row['avg_clicks_per_student_day']
    ax1.text(
        val + 0.5, i,
        f'{val:.1f}',
        va='center', fontsize=9, color='#333333',
    )
ax1.set_yticks(y_pos)
ax1.set_yticklabels(df_eng['code_module'])
ax1.set_xlabel(ENGAGEMENT_LABELS['avg_clicks_per_student_day'])
ax1.set_title('Daily Engagement Intensity')
sns.despine(ax=ax1)

# --- Right: median early clicks (first 28 days) ---
ax2.barh(y_pos, df_eng['median_early_clicks'], color=colors, edgecolor='white')
for i, (_, row) in enumerate(df_eng.iterrows()):
    val = row['median_early_clicks']
    ax2.text(
        val + 10, i,
        f'{val:.0f}',
        va='center', fontsize=9, color='#333333',
    )
ax2.set_yticks(y_pos)
ax2.set_yticklabels(df_eng['code_module'])
ax2.set_xlabel(ENGAGEMENT_LABELS['median_early_clicks'])
ax2.set_title('Early Engagement Volume (first 28 days)')
sns.despine(ax=ax2)

fig.suptitle(
    'Engagement Intensity by Course\n'
    '(modules sorted by completion rate, bottom = lowest)',
    fontsize=14, y=1.02,
)
fig.tight_layout()
save_fig(fig, '06_engagement_by_course')
plt.show()

> **Variazione dell'engagement:** L'intensità di engagement varia tra i corsi, ma la relazione con il completamento non è lineare. Un corso potrebbe avere un alto engagement per sessione (molti click per visita) ma basso completamento se gli studenti alla fine si disimpegnano.
>
> **Due dimensioni dell'engagement:**
> - L'**intensità giornaliera** riflette quanto profondamente gli studenti interagiscono *quando accedono*. Questa è influenzata dal design del corso: corsi ricchi di risorse con elementi interattivi generano naturalmente più click.
> - Il **volume precoce** riflette il momentum iniziale di engagement. BQ2 (NB04) ha mostrato che l'engagement precoce è il predittore più forte del completamento a livello di studente. Qui vediamo se alcuni corsi generano più engagement iniziale di altri.
>
> **Cautela nell'interpretazione:** Le metriche di engagement a livello di corso confondono il design del corso con il comportamento degli studenti. Un corso che attrae studenti più motivati mostrerà un engagement più alto indipendentemente dalla qualità del suo design.

## 6. Heatmap di progettazione del corso

La heatmap standardizza tutte le feature di design e engagement in **z-score** (deviazioni standard dalla media dei 7 moduli) in modo che feature su scale diverse diventino visivamente comparabili.

- Le **celle verdi** indicano valori sopra la media (rispetto agli altri moduli)
- Le **celle rosse** indicano valori sotto la media
- Le **annotazioni** mostrano i valori reali (non standardizzati) come riferimento

I moduli sono ordinati dall'alto verso il basso per tasso di completamento, così i pattern visivi tra i corsi ad alta retention (in alto) e bassa retention (in basso) diventano evidenti.

In [ ]:
# --- Course design heatmap ---
# Z-score standardization puts all features on the same scale for visual
# comparison: how many standard deviations above or below the module mean.
all_features = DESIGN_FEATURES + ENGAGEMENT_FEATURES
all_labels = {**DESIGN_LABELS, **ENGAGEMENT_LABELS}

# Sort by completion rate: highest at top for intuitive reading
module_order = df_bq4.sort_values(
    'avg_completion_rate_pct', ascending=False
)['code_module'].tolist()

df_heat_raw = df_bq4.set_index('code_module').loc[module_order, all_features]

# Z-score: (value - mean) / std across the 7 modules for each feature
df_z = (df_heat_raw - df_heat_raw.mean()) / df_heat_raw.std()

# Readable column labels
readable_cols = [all_labels[c] for c in all_features]
df_z.columns = readable_cols

# Build annotation matrix with actual values (not z-scores)
# so the reader sees real numbers alongside the color encoding
annot_values = []
for _, row in df_heat_raw.iterrows():
    row_annot = []
    for feat in all_features:
        val = row[feat]
        if feat == 'avg_assessment_density':
            row_annot.append(f'{val:.2f}')
        elif feat in ('avg_n_assessments', 'avg_n_activity_types',
                       'avg_clicks_per_student_day'):
            row_annot.append(f'{val:.1f}')
        else:
            row_annot.append(f'{val:.0f}')
    annot_values.append(row_annot)

annot_array = np.array(annot_values)

# --- Heatmap ---
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    df_z, annot=annot_array, fmt='',
    cmap='RdYlGn', center=0, linewidths=0.5,
    ax=ax, cbar_kws={'label': 'Z-score (standard deviations from mean)'},
)

# Add completion rate to y-axis labels for immediate context
rate_labels = []
for m in module_order:
    mask = df_bq4['code_module'] == m
    rate = df_bq4.loc[mask, 'avg_completion_rate_pct'].values[0]
    rate_labels.append(f'{m}  ({rate:.1f}%)')
ax.set_yticklabels(rate_labels, rotation=0)

ax.set_title(
    'Course Design Heatmap — All Features Standardized\n'
    '(z-score coloring, actual values annotated; sorted by completion rate)'
)
ax.set_ylabel('')
fig.tight_layout()
save_fig(fig, '06_course_design_heatmap')
plt.show()

> **Leggere la heatmap:** Cercare pattern di colore verticali (feature che differiscono in modo consistente tra corsi ad alta e bassa retention) e pattern orizzontali (moduli con caratteristiche consistentemente sopra o sotto la media).
>
> **Cosa cercare:**
> - I corsi ad alto completamento (righe in alto) tendono ad essere più verdi su specifiche colonne di feature? Se sì, quelle feature potrebbero contribuire alla retention.
> - I corsi a basso completamento (righe in basso) condividono celle rosse sulle stesse feature? Questo rafforzerebbe il pattern.
> - Ci sono moduli che sono outlier — verdi sulla maggior parte delle feature ma bassi sul completamento, o viceversa? Questi rompono il pattern e suggeriscono altri fattori in gioco.
>
> **Avvertenza:** La standardizzazione in z-score amplifica il contrasto visivo — una differenza di 0.5 deviazioni standard appare drammatica nella scala cromatica ma potrebbe non essere praticamente significativa con solo 7 osservazioni.

## 7. Ipotesi e limitazioni

### Ipotesi (non conclusioni)

Sulla base dei pattern descrittivi osservati:

1. **La struttura delle valutazioni potrebbe influenzare la retention.** Corsi con un certo pacing delle valutazioni potrebbero fornire checkpoint regolari di engagement che aiutano gli studenti a restare in carreggiata — oppure potrebbero creare punti di pressione che spingono all'abbandono. La direzione di questo effetto è una domanda empirica.

2. **La diversità delle risorse potrebbe supportare l'engagement.** Corsi che offrono una maggiore varietà di tipologie di attività VLE potrebbero mantenere gli studenti coinvolti attraverso esperienze di apprendimento diversificate. Tuttavia, la diversità delle risorse potrebbe anche correlare con l'investimento istituzionale nel corso, il che rappresenta un fattore confondente.

3. **L'intensità dell'engagement iniziale varia in base al design del corso.** Alcuni corsi generano un engagement precoce più elevato, che NB04 (BQ2) ha identificato come il predittore più forte a livello di studente per il completamento. Se il design del corso può aumentare l'engagement iniziale, questa è una leva operativa per la retention.

### Limitazioni

Questa analisi presenta vincoli fondamentali che devono essere riconosciuti:

- **n=7.** Sette data point non possono supportare la statistica inferenziale. Tutte le correlazioni, i pattern e le ipotesi sono esplorativi. Un ρ di Spearman significativo per n=7 richiede |ρ| > 0.79 — associazioni estremamente forti.

- **Fallacia ecologica.** Le medie a livello di corso possono mascherare la variazione a livello di studente. Un corso con engagement medio alto potrebbe avere una distribuzione bimodale: molti studenti altamente coinvolti e molti che non hanno mai effettuato l'accesso.

- **Confondimento per materia.** I diversi moduli insegnano materie diverse. La difficoltà intrinseca della materia, la rilevanza professionale e la motivazione degli studenti sono variabili non osservate che potrebbero spiegare sia le scelte di design del corso sia gli esiti di retention.

- **Autoselezione degli studenti.** Gli studenti scelgono quali moduli frequentare. Se studenti più capaci o motivati selezionano determinati corsi, quei corsi mostreranno una retention più alta indipendentemente dal design.

- **Confondimento temporale.** Il design del corso potrebbe essere cambiato tra le presentazioni, e le metriche aggregate appianano questi cambiamenti.

## 8. Conclusioni chiave e prossimi passi

### Cosa abbiamo imparato

1. **I tassi di completamento variano sostanzialmente tra i moduli** — il gap tra i migliori e i peggiori performer è significativo e consistente tra le presentazioni. Questa variazione non è rumore.

2. **Le feature di progettazione del corso mostrano pattern suggestivi** con la retention, ma con sole 7 osservazioni, queste sono ipotesi piuttosto che risultati. La densità delle valutazioni e la diversità delle risorse emergono come candidati per ulteriori indagini.

3. **L'intensità di engagement differisce per corso.** Alcuni moduli generano più interazione VLE di altri. Poiché l'engagement precoce è il predittore più forte del completamento a livello di studente (BQ2), design dei corsi che promuovono un engagement iniziale più elevato potrebbero supportare indirettamente la retention.

4. **La heatmap rivela profili dei corsi** — i moduli differiscono non solo nel tasso di completamento ma nelle loro caratteristiche complessive di design e engagement. I corsi ad alta retention potrebbero condividere certi pattern di progettazione.

5. **Nessuna affermazione causale è possibile** con questa dimensione campionaria. I pattern osservati qui sono input per le raccomandazioni di BQ5, non evidenze autonome.

### Prossimi passi

| Notebook | Business Question | Focus |
|----------|-------------------|-------------------------------------------------|
| **07** | BQ5 | Le 3 principali raccomandazioni operative per la retention |

NB07 sintetizza i risultati di tutte e cinque le business question (BQ1–BQ4) in raccomandazioni concrete e prioritizzate per un operatore di piattaforma — l'output analitico finale di questo progetto.

---

**Riproducibilità:** Tutte le figure sono salvate in `reports/figures/`. Per riprodurre questo notebook, eseguire prima `python -m run_pipeline`, poi eseguire tutte le celle in ordine.

> **Dai profili dei corsi all'azione:** Questo notebook ha caratterizzato *come i corsi differiscono* in design, engagement e retention. Combinato con gli insight a livello di studente da BQ1–BQ3, ora abbiamo il quadro completo necessario per rispondere alla domanda finale: cosa può fare concretamente un operatore di piattaforma?
>
> Continua con il **Notebook 07** (`07_bq5_recommendations_synthesis.ipynb`) per BQ5: le 3 principali raccomandazioni operative per migliorare la retention degli studenti.